# TeenPattiBench — evaluate the fine-tuned Llama 3.2 3B on the 2,000-question benchmark

Scores the LoRA adapter from the fine-tuning notebook on **both** benchmark files
(`teenpattibench_eval.jsonl` and `teenpattibench_eval_shuffled.jsonl`), in exactly the
output format of the batch eval: a metadata header plus `{"id", "answer"}` lines, and a
`metrics.py` report per file.

It runs two models, both on this GPU:

| run | what | file slug |
|---|---|---|
| **base** | plain Llama 3.2 3B, with the **same prompt as training** | `llama3.2_3b_base_ftprompt` |
| **fine-tuned** | the same model + `best_adapter/` | `llama3.2_3b_ft` |

The base run matters: the batch eval rendered Llama's chat template with *that day's*
date, while fine-tuning fixed it to `26 Jul 2024`. Running the base model here with the
fine-tuning prompt makes the comparison differ in **one thing only — the adapter**.

**This notebook trains nothing.** The eval is only read, generated on and scored.

### Before you run
1. **Runtime → Change runtime type → L4** (or A100). It needs bfloat16, so not a T4.
2. **🔑 Secrets:** `HF_TOKEN` with notebook access (the base Llama weights are gated).
3. **📁 Upload into `/content`:**
   `teenpattibench_eval.jsonl` · `teenpattibench_eval_shuffled.jsonl` ·
   `teenpattibench_system_prompt.txt` · `metrics.py`
4. **The adapter** — nothing to do if the fine-tuning run's Drive folder is still there
   (`MyDrive/teenpattibench_finetune/llama3.2_3b_full/best_adapter`). Otherwise upload the
   downloaded `llama3.2_3b_full_finetune_results.zip` to `/content`; it is unpacked for you.
5. *Optional:* upload the earlier batch-run files `preds_llama3.2_3b_colab.jsonl` and
   `preds_llama3.2_3b_colab_shuffled.jsonl` and they are added to the summary table.
6. **Runtime → Run all.** Allow Google Drive. Results are saved there as each file
   finishes; if Colab disconnects, **Run all** again skips what is done.

Time on an L4: roughly 10–20 min for all four runs.

## 1. Configuration

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao"])

In [ ]:
BASE_ID   = "meta-llama/Llama-3.2-3B-Instruct"
BASE_NAME = "Llama 3.2 3B"
TAG       = "llama3.2_3b"

# Where the adapter may be. The first folder containing adapter_config.json is used.
ADAPTER_CANDIDATES = [
    "/content/drive/MyDrive/teenpattibench_finetune/llama3.2_3b_full/best_adapter",
    "/content/best_adapter",
    "/content/llama3.2_3b_full_finetune_results/best_adapter",
]
ADAPTER_ZIP = "/content/llama3.2_3b_full_finetune_results.zip"   # unpacked if no folder found

RUN_BASE = True          # base Llama with the training prompt, for a like-for-like comparison

EVALS = [                # same two files, same order, as the batch eval
    ("v1",       "teenpattibench_eval.jsonl",          ""),
    ("shuffled", "teenpattibench_eval_shuffled.jsonl", "_shuffled"),
]

# Must match training exactly -- the adapter learned this prompt.
DATE_STRING    = "26 Jul 2024"
DTYPE          = "bfloat16"
BATCH_SIZE     = 8
MAX_NEW_TOKENS = 16
LIMIT          = None     # e.g. 20 for a quick end-to-end trial (results then not saved as final)

DATA_DIR       = "/content"
DRIVE_DATA_DIR = "/content/drive/MyDrive/teenpattibench_data"   # only for the optional leak check
OUT_DIR        = "/content/drive/MyDrive/teenpattibench_results_finetuned"

## 2. Packages and GPU

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "accelerate"])

import torch, transformers, peft, os, json, re, gc, time, math, shutil, hashlib, datetime, collections
from packaging import version
print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> L4.")
if not torch.cuda.is_bf16_supported():
    raise SystemExit("This GPU has no bfloat16 (a T4?). The adapter was trained in bf16: use an L4 or A100.")
TORCH_DTYPE = getattr(torch, DTYPE)
GPU = torch.cuda.get_device_name(0)
print(f"GPU {GPU} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Google Drive and output folder

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if LIMIT:
    OUT_DIR = OUT_DIR + f"_trial{LIMIT}"      # a trial never mixes with the real results
os.makedirs(OUT_DIR, exist_ok=True)
done = sorted(f for f in os.listdir(OUT_DIR) if f.startswith("preds_"))
print(f"results -> {OUT_DIR}")
print(f"already finished: {done or 'nothing'}")

## 4. Load the uploaded eval files and scorer

In [ ]:
need = [f for _, f, _ in EVALS] + ["teenpattibench_system_prompt.txt", "metrics.py"]
missing = [f for f in need if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise SystemExit(f"Missing from {DATA_DIR}: {missing}\nUpload them with the Files panel. "
                     "If one shows up as 'name (1).ext', rename it or delete the duplicate.")
sys.path.insert(0, DATA_DIR)
import metrics

SYSTEM_PATH = os.path.join(DATA_DIR, "teenpattibench_system_prompt.txt")
SYSTEM_PROMPT = open(SYSTEM_PATH, encoding="utf-8").read()
EVAL_ITEMS = {}
for tag, fname, _ in EVALS:
    items = [json.loads(l) for l in open(os.path.join(DATA_DIR, fname), encoding="utf-8") if l.strip()]
    if len(items) != 2000:
        raise SystemExit(f"{fname}: expected 2,000 items, found {len(items):,}")
    EVAL_ITEMS[tag] = items[:LIMIT] if LIMIT else items
    print(f"{tag:<9} {fname:<38} {len(EVAL_ITEMS[tag]):,} items")
if [i["id"] for i in EVAL_ITEMS["v1"]] != [i["id"] for i in EVAL_ITEMS["shuffled"]]:
    raise SystemExit("v1 and shuffled files do not have the same ids in the same order.")
print(f"system prompt {len(SYSTEM_PROMPT)} chars")

## 5. Find and check the adapter

Checks it was trained on **this** base model, records which step it is and a SHA-256 of its
weights, and confirms the training run declared `benchmark_eval_used: false`.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_adapter():
    for d in ADAPTER_CANDIDATES:
        if os.path.exists(os.path.join(d, "adapter_config.json")):
            return d
    return None

ADAPTER_DIR = find_adapter()
if ADAPTER_DIR is None and os.path.exists(ADAPTER_ZIP):
    import zipfile
    dest = "/content/llama3.2_3b_full_finetune_results"
    with zipfile.ZipFile(ADAPTER_ZIP) as z:
        z.extractall(dest)
    print(f"unpacked {ADAPTER_ZIP} -> {dest}")
    ADAPTER_DIR = find_adapter()
if ADAPTER_DIR is None:
    raise SystemExit("No adapter found. Looked in:\n  " + "\n  ".join(ADAPTER_CANDIDATES) +
                     f"\nand no {ADAPTER_ZIP}. Upload the fine-tuning zip or fix ADAPTER_CANDIDATES.")

acfg = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json")))
if acfg.get("base_model_name_or_path") != BASE_ID:
    raise SystemExit(f"adapter was trained on {acfg.get('base_model_name_or_path')!r}, not {BASE_ID!r}")
wfile = next((os.path.join(ADAPTER_DIR, f) for f in ("adapter_model.safetensors", "adapter_model.bin")
              if os.path.exists(os.path.join(ADAPTER_DIR, f))), None)
if wfile is None:
    raise SystemExit(f"{ADAPTER_DIR} has adapter_config.json but no adapter weights.")

BEST = json.load(open(os.path.join(ADAPTER_DIR, "best.json"))) if os.path.exists(
    os.path.join(ADAPTER_DIR, "best.json")) else {}
RUN_CFG = json.load(open(os.path.join(ADAPTER_DIR, "run_config.json"))) if os.path.exists(
    os.path.join(ADAPTER_DIR, "run_config.json")) else {}
if RUN_CFG and RUN_CFG.get("data", {}).get("benchmark_eval_used") is not False:
    raise SystemExit("run_config.json does not say benchmark_eval_used: false -- wrong adapter?")
if RUN_CFG.get("prompt", {}).get("date_string", DATE_STRING) != DATE_STRING:
    raise SystemExit(f"adapter trained with date_string {RUN_CFG['prompt']['date_string']!r}; "
                     f"set DATE_STRING to match.")

ADAPTER_INFO = {
    "adapter_dir": ADAPTER_DIR, "adapter_weights_sha256": sha256(wfile),
    "base_model": acfg.get("base_model_name_or_path"),
    "lora": {"r": acfg.get("r"), "alpha": acfg.get("lora_alpha"),
             "dropout": acfg.get("lora_dropout"), "targets": sorted(acfg.get("target_modules") or [])},
    "best_step": BEST.get("step"), "val_acc": BEST.get("acc"), "val_balanced_acc": BEST.get("balanced_acc"),
    "train_mode": RUN_CFG.get("mode"), "train_seed": RUN_CFG.get("seed"),
    "finetune_file_sha256": RUN_CFG.get("data", {}).get("finetune_sha256"),
    "benchmark_eval_used_in_training": RUN_CFG.get("data", {}).get("benchmark_eval_used"),
    "trained_with": RUN_CFG.get("runtime"),
}
print(f"adapter: {ADAPTER_DIR}")
for k, v in ADAPTER_INFO.items():
    print(f"  {k:<32} {v}")
if not RUN_CFG:
    print("  (no run_config.json next to the adapter -- provenance fields above are empty)")

## 6. Optional: prove the eval never appeared in the fine-tuning data

Runs only if `teenpattibench_finetune.jsonl` is in `/content` or in the Drive data folder
used for fine-tuning. Checks the same three things as the fine-tuning split: identical
prompts, the same seen hand at the same betting position, and the same blind situation.

In [ ]:
ft_path = next((p for p in (os.path.join(DATA_DIR, "teenpattibench_finetune.jsonl"),
                            os.path.join(DRIVE_DATA_DIR, "teenpattibench_finetune.jsonl"))
                if os.path.exists(p)), None)
LEAK_CHECK = {"ran": False}
if ft_path is None:
    print("teenpattibench_finetune.jsonl not found -- leak check skipped (it is optional).")
else:
    fp, fk, fb = set(), set(), set()
    for l in open(ft_path, encoding="utf-8"):
        r = json.loads(l); m = r["meta"]
        fp.add(r["prompt"])
        (fb.add(m["node"]) if m["blind"] else fk.add((m["node"], m["iso_class"])))
    LEAK_CHECK = {"ran": True, "finetune_file": ft_path}
    for tag, items in EVAL_ITEMS.items():
        c = {"prompt": sum(i["prompt"] in fp for i in items),
             "seen_infoset": sum((i["meta"]["node"], i["meta"]["iso_class"]) in fk
                                 for i in items if not i["meta"]["blind"]),
             "blind_situation": sum(i["meta"]["node"] in fb for i in items if i["meta"]["blind"])}
        LEAK_CHECK[tag] = c
        print(f"{tag:<9} overlap with fine-tuning data: {c}")
        if any(c.values()):
            raise SystemExit("LEAK: eval questions appear in the fine-tuning data. Stop.")
    print("no overlap -- the eval is unseen by the fine-tuned model.")

## 7. Hugging Face login, tokenizer and prompt (identical to fine-tuning)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(BASE_ID)
PAD_ID = tok.convert_tokens_to_ids("<|finetune_right_pad_id|>")
if PAD_ID is None or PAD_ID == tok.unk_token_id:
    PAD_ID = tok.eos_token_id
tok.pad_token_id = PAD_ID
tok.padding_side = "left"

def build_prompt(user_text):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False, date_string=DATE_STRING)

p = build_prompt("probe")
assert p.count("<|begin_of_text|>") == 1, "BOS should appear exactly once"
assert DATE_STRING in p, "date_string not applied -- template changed?"
print(p.replace(SYSTEM_PROMPT, "<SYSTEM PROMPT>"))

## 8. Helpers

Answer extraction, output format and scoring are the batch eval's, unchanged: the first
action word in the completion, matched against the full vocabulary (not the legal set).

In [ ]:
from tqdm.auto import tqdm
ACTION_WORDS = ("stay-blind", "see", "chaal", "raise", "pack", "show")
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56.0") else "torch_dtype"

def extract_answer(completion):
    text = completion.strip().lower()
    hits = [(m.start(), w) for w in ACTION_WORDS
            for m in re.finditer(rf"(?<![a-z-]){re.escape(w)}(?![a-z-])", text)]
    return min(hits)[1] if hits else text

def load_base():
    m = AutoModelForCausalLM.from_pretrained(BASE_ID, device_map={"": 0}, **{DTYPE_KW: TORCH_DTYPE})
    gc_ = m.generation_config
    gc_.do_sample = False
    for k in ("temperature", "top_p", "top_k"):
        if hasattr(gc_, k):
            setattr(gc_, k, None)
    m.eval()
    return m

def inner(m):
    return m.get_base_model() if hasattr(m, "get_base_model") else m

@torch.no_grad()
def generate_all(m, items, desc):
    out = []
    for s in tqdm(range(0, len(items), BATCH_SIZE), desc=desc):
        batch = items[s:s + BATCH_SIZE]
        enc = tok([build_prompt(i["prompt"]) for i in batch], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(m.device)
        g = m.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                       pad_token_id=PAD_ID, use_cache=True)
        for it, t in zip(batch, tok.batch_decode(g[:, enc["input_ids"].shape[1]:],
                                                 skip_special_tokens=True)):
            out.append({"id": it["id"], "answer": extract_answer(t)})
    return out

def metadata(m, name, eval_path, n_items, pred_path, finetuned):
    b = inner(m)
    meta = {
        "type": "metadata",
        "model": {"name": name, "model_id": BASE_ID, "architecture": list(b.config.architectures),
                  "model_type": b.config.model_type,
                  "num_parameters": int(sum(p.numel() for p in b.parameters())),
                  "dtype": str(TORCH_DTYPE), "device": "cuda:0", "quantized": False},
        "tokenizer": {"name_or_path": tok.name_or_path, "vocab_size": tok.vocab_size,
                      "pad_token": tok.convert_ids_to_tokens(PAD_ID), "eos_token": tok.eos_token,
                      "bos_token": tok.bos_token},
        "generation": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "temperature": None,
                       "top_p": None, "top_k": None, "thinking_enabled": False},
        "dataset": {"name": "TeenPattiBench", "evaluation_file": eval_path, "num_questions": n_items},
        "prompt": {"system_prompt_file": SYSTEM_PATH, "system_prompt": SYSTEM_PROMPT,
                   "date_string": DATE_STRING},
        "runtime": {"pytorch_version": torch.__version__, "cuda_available": True,
                    "cuda_version": torch.version.cuda, "gpu_count": torch.cuda.device_count(),
                    "gpus": [GPU], "transformers_version": transformers.__version__,
                    "peft_version": peft.__version__},
        "output": {"format": "JSONL", "prediction_file": pred_path},
    }
    if finetuned:
        meta["finetune"] = ADAPTER_INFO
    return meta

def write_preds(path, meta, records):
    tmp = path + ".part"                     # atomic: a half-written file is never "done"
    with open(tmp, "w", encoding="utf-8") as fh:
        fh.write(json.dumps(meta) + "\n")
        for r in records:
            fh.write(json.dumps(r) + "\n")
    os.replace(tmp, path)

def score_and_save(items, pred_path):
    recs = [json.loads(l) for l in open(pred_path, encoding="utf-8")][1:]
    rep = metrics.score(items, {r["id"]: {"action": r["answer"]} for r in recs})
    stem = os.path.join(OUT_DIR, os.path.basename(pred_path).replace("preds_", "report_").replace(".jsonl", ""))
    json.dump(rep, open(stem + ".json", "w"), indent=1)
    open(stem + ".txt", "w").write(metrics.format_report(rep, metrics.trivial_baselines(items)) + "\n")
    return rep

def pred_path(slug, suf):
    return os.path.join(OUT_DIR, f"preds_{slug}_colab{suf}.jsonl")

RUNS = ([("base", f"{BASE_NAME} (training prompt)", f"{TAG}_base_ftprompt")] if RUN_BASE else []) + \
       [("ft", f"{BASE_NAME} + TeenPatti LoRA", f"{TAG}_ft")]
print("runs:", [r[2] for r in RUNS])

## 9. Run

Loads the base model once, runs the base evals, then puts the adapter on top and runs the
fine-tuned evals. Each file is saved to Drive and scored as soon as it finishes.

In [ ]:
LOG = os.path.join(OUT_DIR, "run_log.jsonl")
def log(**kw):
    kw["time_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with open(LOG, "a") as fh:
        fh.write(json.dumps(kw) + "\n")

todo = [(k, n, s, tag, f, suf) for k, n, s in RUNS for tag, f, suf in EVALS
        if not os.path.exists(pred_path(s, suf))]
if not todo:
    print("everything already done -- go to the summary.")
else:
    t_all = time.time()
    model = load_base()
    print(f"base model loaded | GPU memory {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")
    have_adapter = False
    for kind, name, slug, tag, fname, suf in todo:
        if kind == "ft" and not have_adapter:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, ADAPTER_DIR)
            model.eval()
            # the adapter must really be loaded: LoRA B matrices start at zero
            lb = [p for n_, p in model.named_parameters() if "lora_B" in n_]
            nz = sum(int(p.abs().sum().item() > 0) for p in lb)
            print(f"adapter loaded: {len(lb)} LoRA-B matrices, {nz} non-zero")
            if not lb or nz == 0:
                raise SystemExit("adapter weights are all zero -- this is an untrained adapter.")
            have_adapter = True
        items, path = EVAL_ITEMS[tag], pred_path(slug, suf)
        smoke = [r["answer"] for r in generate_all(model, items[:5], f"{slug} smoke")]
        print(f"smoke test {slug}: {smoke}")
        if not any(a in ACTION_WORDS for a in smoke):
            raise SystemExit(f"{slug}: no action word in 5 completions -- stopping.")
        t1 = time.time()
        recs = generate_all(model, items, f"{slug} {tag}")
        write_preds(path, metadata(model, name, os.path.join(DATA_DIR, fname), len(items), path,
                                   kind == "ft"), recs)
        rep = score_and_save(items, path)
        aa = 100 * rep["overall"]["aa"] / rep["overall"]["n"]
        print(f"  {slug} {tag}: {len(recs):,} answers in {(time.time() - t1) / 60:.1f} min "
              f"-> Action Accuracy {aa:.2f}%")
        log(run=slug, eval=tag, status="done", action_accuracy=aa, minutes=round((time.time() - t1) / 60, 2))
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"\nALL RUNS FINISHED in {(time.time() - t_all) / 60:.1f} min")

## 10. Summary

Every stratum is shown next to what answering one word forever would score there, which is
the only fair comparison. **Conversion is broken down by turn and never averaged across
turns**: the solver's pattern is no / yes / no / yes, so an average hides exactly what is
being tested. Turn 1 rests on 4 items and is a control, not a result.

*Paired* comparisons use the same 1,800 graded questions for both runs: exact McNemar test.

In [ ]:
import csv
ALL = ACTION_WORDS

def strata(items):
    g = [i for i in items if not i["is_mixed"]]
    return {"all": g,
            "seen": [i for i in g if i["meta"]["family"] == "seen_bet"],
            "conversion": [i for i in g if i["meta"]["family"] == "look"],
            "blind_bet": [i for i in g if i["meta"]["family"] == "blind_bet"]}

def best_const(sub):
    return max(100 * sum(i["correct_action"] == a for i in sub) / len(sub) for a in ALL) if sub else float("nan")

def mcnemar(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(n, j) for j in range(k + 1)) / 2 ** n)

def load_preds(p):
    L = [json.loads(l) for l in open(p, encoding="utf-8")]
    return {r["id"]: r["answer"] for r in L[1:]}

SOURCES = []    # (label, slug, folder)
if all(os.path.exists(os.path.join(DATA_DIR, f"preds_{TAG}_colab{s}.jsonl")) for _, _, s in EVALS):
    SOURCES.append(("base, batch run (run-day date)", TAG, DATA_DIR))
SOURCES += [(("base, training prompt" if k == "base" else "FINE-TUNED"), s, OUT_DIR) for k, _, s in RUNS]

S1 = strata(EVAL_ITEMS["v1"])
BASE_LINE = {k: best_const(v) for k, v in S1.items()}
print(f"graded items: all {len(S1['all'])} | seen {len(S1['seen'])} | conversion {len(S1['conversion'])} "
      f"| blind_bet {len(S1['blind_bet'])}   (200 mixed items are TVD-only, not graded)\n")
print(f"{'':<32}{'eval':<10}{'all':>7}{'seen':>7}{'conv':>7}{'blind':>7}   conv turn 1 / 2 / 3 / 4    blind fold")
print(f"{'one word forever':<32}{'':<10}" + "".join(f"{BASE_LINE[k]:7.2f}" for k in ("all", "seen", "conversion", "blind_bet")))

rows, P, answers = [], {}, {}
for label, slug, folder in SOURCES:
    for tag, _, suf in EVALS:
        p = os.path.join(folder, f"preds_{slug}_colab{suf}.jsonl")
        if not os.path.exists(p):
            continue
        pr = load_preds(p); P[(slug, tag)] = pr
        S = strata(EVAL_ITEMS[tag])
        acc = {k: 100 * sum(pr[i["id"]] == i["correct_action"] for i in v) / len(v) for k, v in S.items() if v}
        turn = {}
        for t in (1, 2, 3, 4):
            s_ = [i for i in S["conversion"] if i["meta"]["turn_index"] == t]
            turn[t] = (100 * sum(pr[i["id"]] == i["correct_action"] for i in s_) / len(s_), len(s_)) if s_ else (float("nan"), 0)
        bfold = 100 * sum(pr[i["id"]] == "pack" for i in S["blind_bet"]) / max(1, len(S["blind_bet"]))
        conv_ans = dict(collections.Counter(pr[i["id"]] for i in S["conversion"]).most_common())
        answers[f"{slug}_{tag}"] = {"all": dict(collections.Counter(pr.values()).most_common()),
                                    "conversion": conv_ans}
        rows.append({"model": label, "slug": slug, "eval": tag,
                     **{f"aa_{k}": round(v, 2) for k, v in acc.items()},
                     **{f"conv_turn{t}": round(v[0], 2) for t, v in turn.items()},
                     **{f"conv_turn{t}_n": v[1] for t, v in turn.items()},
                     "blind_fold_pct": round(bfold, 2), "conversion_answers": conv_ans})
        ts = " / ".join(f"{v[0]:5.1f}" for v in turn.values())
        print(f"{label:<32}{tag:<10}" + "".join(f"{acc.get(k, float('nan')):7.2f}"
              for k in ("all", "seen", "conversion", "blind_bet")) + f"   {ts}    {bfold:5.1f}%")

print("\nconversion turn sizes:", {t: sum(i['meta']['turn_index'] == t for i in S1['conversion']) for t in (1, 2, 3, 4)})
print("conversion answers:")
for k, v in answers.items():
    flag = "   <- ONE answer for every item: no timing" if len(v["conversion"]) == 1 else ""
    print(f"  {k:<40} {v['conversion']}{flag}")

# ---- paired tests on the same graded questions --------------------------------
PAIRS = []
ft_slug = f"{TAG}_ft"
for other_label, other in [(l, s) for l, s, _ in SOURCES if s != ft_slug]:
    for tag, _, _ in EVALS:
        if (ft_slug, tag) in P and (other, tag) in P:
            PAIRS.append((f"fine-tuned vs {other_label}", tag, P[(ft_slug, tag)], P[(other, tag)], tag))
if (ft_slug, "v1") in P and (ft_slug, "shuffled") in P:
    PAIRS.append(("fine-tuned: shuffled vs v1", "both", P[(ft_slug, "shuffled")], P[(ft_slug, "v1")], "v1"))
paired = []
print("\npaired comparisons (graded items):")
for name, tag, a, b, keytag in PAIRS:
    g = strata(EVAL_ITEMS[keytag])["all"]
    ra = {i["id"] for i in g if a[i["id"]] == i["correct_action"]}
    rb = {i["id"] for i in g if b[i["id"]] == i["correct_action"]}
    only_a, only_b = len(ra - rb), len(rb - ra)
    pv = mcnemar(only_a, only_b)
    d = 100 * (len(ra) - len(rb)) / len(g)
    paired.append({"comparison": name, "eval": tag, "delta_pts": round(d, 2), "only_first_right": only_a,
                   "only_second_right": only_b, "mcnemar_p": pv})
    print(f"  {name:<52} {tag:<9} {d:+7.2f} pts   right only in first {only_a:4d}, only in second "
          f"{only_b:4d}   McNemar p = {pv:.2g}")

print("\nfor reference (zero-shot, batch eval, bf16): best of nine = Gemma 3 12B 36.17% (shuffled 34.22%);"
      "\nLlama 3.2 3B 26.78% (26.39%). Those used each model's own default prompt.")

keys = sorted({k for r in rows for k in r if k != "conversion_answers"},
              key=lambda k: (k not in ("model", "slug", "eval"), k))
with open(os.path.join(OUT_DIR, "finetuned_summary.csv"), "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=keys, extrasaction="ignore"); w.writeheader(); w.writerows(rows)
json.dump({"baselines": BASE_LINE, "rows": rows, "paired": paired, "answer_distributions": answers,
           "adapter": ADAPTER_INFO, "leak_check": LEAK_CHECK,
           "prompt": {"date_string": DATE_STRING, "system_prompt_file": SYSTEM_PATH}},
          open(os.path.join(OUT_DIR, "finetuned_summary.json"), "w"), indent=1)
print(f"\nsaved finetuned_summary.csv and finetuned_summary.json to {OUT_DIR}")

## 11. Download everything as one zip

In [ ]:
import zipfile
zpath = "/content/teenpattibench_finetuned_eval_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        if f.endswith((".jsonl", ".json", ".txt", ".csv")) and not f.endswith(".part"):
            z.write(os.path.join(OUT_DIR, f), f)
with zipfile.ZipFile(zpath) as z:
    names = z.namelist()
print(f"{len(names)} files zipped ({os.path.getsize(zpath) / 1e6:.1f} MB):")
for n in names:
    print("  ", n)
from google.colab import files
files.download(zpath)